# HPDM097 Assignment 2

## Iteration 2: Acute beds (Gemini)

This is the 2nd iterations to the code. Added acute beds as a resource and a length of stay (LOS) logic for each patient type. Also added to the rest of the patient pathway from the acute ward so patient can be transferred to either a rehab ward, early supported discahrge (ESD) or 'Other' which includes own home, care home or mortality

Prompt and response: https://gemini.google.com/share/9633688a34a2

## Prompt:
Yes, let's still follow the iterative approach and recreating the code above but now add the capacity resource, and length of stay logic.

There are 10 acute stroke unit beds which act as the resource.

To expand the logic diagram, after the acute stroke ward with the resource of 10 acute stroke unit beds, patients can then be sent to three places: rehab ward, Early supported discharge (ESD) or Other (any destination otters than rehab or ESD including own home, care home or mortality)

The acute ward length of stay (LOS) follows a log-normal distribution with the following parameters in days:
Acute stroke (No ESD) - mean: 7.4, stdev: 8.6
Acute stroke (ESD) - mean: 4.6, stdev: 4.8
Acute stroke (mortality) - mean: 7.0, stdev: 8.7
TIA - mean: 1.8, stdev: 2.3
Complex neurological - mean: 4.0, stdev: 5.0
Other - mean: 3.8, stdev: 5.2

Here is the transfer matrix from the acute stroke unit:
Stroke patients: 24% to rehab, 13% to ESD, 63% to other
TIA patients: 1% to rehab, 1% to ESD, 98% to other
Complex neurological patients: 11% to rehab, 5% to ESD, 84% to other
Other patients: 5% to rehab, 10% to ESD, 85% to other

## Response:
This is a perfect next step. We are introducing two core components of Discrete Event Simulation: Resources (the limited number of beds) and Routing (the pathways patients take).

A critical note on the log-normal distribution: The parameters you provided are the sample mean and standard deviation. Python’s numpy.random.lognormal function requires the mean and standard deviation of the underlying normal distribution. I have updated the LognormalDistribution class to automatically perform this mathematical conversion for you so the samples correctly reflect your real-world data.

Since the transfer matrix only specifies "Other" (and not a specific "Mortality" split), I have mapped the "Acute stroke (No ESD)" length of stay parameters to both the Rehab and Other pathways for now, while storing the mortality parameters in comments for future use.

Here is the updated, expanded code:

In [1]:
import simpy
import numpy as np

# ---------------------------------------------------------
# 2. Trace utility function as a debug print switch
# ---------------------------------------------------------
TRACE = False

def trace(msg):
    """Utility function to print debug messages if TRACE is True."""
    if TRACE:
        print(msg)

# ---------------------------------------------------------
# 1. Build and sample from distribution classes
# ---------------------------------------------------------
class ExponentialDistribution:
    """Class to sample from an exponential distribution for arrivals."""
    def __init__(self, mean, random_seed=None):
        self.mean = mean
        self.rng = np.random.default_rng(seed=random_seed)
        
    def sample(self):
        return self.rng.exponential(scale=self.mean)

class LognormalDistribution:
    """Class to sample from a lognormal distribution for length of stay."""
    def __init__(self, sample_mean, sample_stdev, random_seed=None):
        # Convert sample mean and stdev to underlying normal distribution parameters
        # This is mathematically required for np.random.lognormal
        variance = sample_stdev ** 2
        self.mu = np.log((sample_mean ** 2) / np.sqrt(variance + sample_mean ** 2))
        self.sigma = np.sqrt(np.log(1 + (variance / sample_mean ** 2)))
        self.rng = np.random.default_rng(seed=random_seed)
        
    def sample(self):
        return self.rng.lognormal(mean=self.mu, sigma=self.sigma)

# ---------------------------------------------------------
# 3. Create a parameter container class called Scenario
# ---------------------------------------------------------
# 6. Input and specify the model parameters as above
# ---------------------------------------------------------
class Scenario:
    """Contains all parameters and configurations for the simulation."""
    def __init__(self):
        self.random_seed = 42
        
        # Resource Capacity
        self.acute_beds_capacity = 10
        
        # Mean inter-arrival times (in days)
        self.inter_arrival_means = {
            'Acute stroke': 1.2,
            'TIA': 9.3,
            'Complex neurological': 3.6,
            'Other': 3.2
        }
        
        # Transfer Matrix Probabilities: [Rehab, ESD, Other]
        self.destinations = ['Rehab', 'ESD', 'Other']
        self.transfer_matrix = {
            'Acute stroke': [0.24, 0.13, 0.63],
            'TIA': [0.01, 0.01, 0.98],
            'Complex neurological': [0.11, 0.05, 0.84],
            'Other': [0.05, 0.10, 0.85]
        }
        
        # Length of Stay Parameters (sample mean, sample stdev) mapped to destinations
        # Note: 'Mortality' (7.0, 8.7) is omitted from routing for now, 'No ESD' used for 'Other' and 'Rehab'
        self.los_params = {
            'Acute stroke': {
                'Rehab': (7.4, 8.6),  
                'ESD': (4.6, 4.8),    
                'Other': (7.4, 8.6)   
            },
            'TIA': {dest: (1.8, 2.3) for dest in self.destinations},
            'Complex neurological': {dest: (4.0, 5.0) for dest in self.destinations},
            'Other': {dest: (3.8, 5.2) for dest in self.destinations}
        }

# ---------------------------------------------------------
# 4. Create a patient class called AcutePatient
# ---------------------------------------------------------
class AcutePatient:
    """Represents an individual patient flowing through the system."""
    def __init__(self, p_id, p_type, arrival_time):
        self.id = p_id
        self.type = p_type
        self.arrival_time = arrival_time
        
        # Attributes populated during flow
        self.destination = None
        self.bed_request_time = None
        self.bed_acquire_time = None
        self.length_of_stay = None
        self.discharge_time = None

# ---------------------------------------------------------
# 5. Create a model class called StrokeUnit
# ---------------------------------------------------------
class StrokeUnit:
    """The main simulation model encapsulating the environment and logic."""
    def __init__(self, env, scenario):
        self.env = env
        self.scenario = scenario
        self.patient_counter = 0
        
        # Initialize the Acute Bed Resource
        self.acute_beds = simpy.Resource(env, capacity=self.scenario.acute_beds_capacity)
        
        # Initialize RNG for routing (seed offset for independence)
        self.routing_rng = np.random.default_rng(seed=self.scenario.random_seed + 999)
        
        # Initialize arrival and LOS distributions
        self.arrival_distributions = {}
        self.los_distributions = {p_type: {} for p_type in self.scenario.inter_arrival_means.keys()}
        
        dist_seed_offset = 0
        for p_type, mean_iat in self.scenario.inter_arrival_means.items():
            self.arrival_distributions[p_type] = ExponentialDistribution(
                mean=mean_iat, random_seed=self.scenario.random_seed + dist_seed_offset
            )
            dist_seed_offset += 1
            
            # Setup LOS distributions per patient type and destination
            for dest in self.scenario.destinations:
                mean, stdev = self.scenario.los_params[p_type][dest]
                self.los_distributions[p_type][dest] = LognormalDistribution(
                    sample_mean=mean, sample_stdev=stdev, 
                    random_seed=self.scenario.random_seed + dist_seed_offset
                )
                dist_seed_offset += 1

    def arrivals_generator(self, patient_type):
        """Generates patient arrivals for a specific patient subtype."""
        while True:
            inter_arrival_time = self.arrival_distributions[patient_type].sample()
            yield self.env.timeout(inter_arrival_time)
            
            self.patient_counter += 1
            patient = AcutePatient(p_id=self.patient_counter, p_type=patient_type, arrival_time=self.env.now)
            
            trace(f"Day {self.env.now:06.2f} | Patient {patient.id:04d} ({patient.type}) arrived.")
            
            # Start the patient pathway process for this new patient
            self.env.process(self.patient_pathway(patient))

    def patient_pathway(self, patient):
        """Handles the flow of a patient requesting a bed, staying, and discharging."""
        # Determine discharge destination early
        probs = self.scenario.transfer_matrix[patient.type]
        patient.destination = self.routing_rng.choice(self.scenario.destinations, p=probs)
        
        patient.bed_request_time = self.env.now
        
        # Request an acute bed
        with self.acute_beds.request() as req:
            yield req  # Yield until a bed is available (acts as a queue)
            
            patient.bed_acquire_time = self.env.now
            wait_time = patient.bed_acquire_time - patient.bed_request_time
            trace(f"Day {self.env.now:06.2f} | Patient {patient.id:04d} ({patient.type}) secured a bed. Wait time: {wait_time:.2f} days.")
            
            # Sample Length of Stay
            los = self.los_distributions[patient.type][patient.destination].sample()
            patient.length_of_stay = los
            
            # Occupy the bed for the LOS
            yield self.env.timeout(los)
            
            patient.discharge_time = self.env.now
            trace(f"Day {self.env.now:06.2f} | Patient {patient.id:04d} ({patient.type}) discharged to {patient.destination}. LOS: {patient.length_of_stay:.2f} days.")

    def run_simulation(self):
        """Starts the arrival generators for all patient subtypes."""
        for p_type in self.scenario.inter_arrival_means.keys():
            self.env.process(self.arrivals_generator(p_type))

# =========================================================
# Execution / Testing Block
# =========================================================
if __name__ == "__main__":
    # Temporarily set to True to verify flow and routing
    TRACE = True  
    
    print("--- Starting Acute Stroke Ward Simulation ---")
    scenario = Scenario()
    env = simpy.Environment()
    model = StrokeUnit(env, scenario)
    model.run_simulation()
    
    # Run for 30 days to observe some queues and discharges
    env.run(until=30)
    print(f"\nSimulation complete. Total arrivals processed: {model.patient_counter}")

--- Starting Acute Stroke Ward Simulation ---
Day 000.51 | Patient 0001 (Other) arrived.
Day 000.51 | Patient 0001 (Other) secured a bed. Wait time: 0.00 days.
Day 002.89 | Patient 0002 (Acute stroke) arrived.
Day 002.89 | Patient 0002 (Acute stroke) secured a bed. Wait time: 0.00 days.
Day 003.66 | Patient 0001 (Other) discharged to Other. LOS: 3.15 days.
Day 005.37 | Patient 0003 (Complex neurological) arrived.
Day 005.37 | Patient 0003 (Complex neurological) secured a bed. Wait time: 0.00 days.
Day 005.69 | Patient 0004 (Acute stroke) arrived.
Day 005.69 | Patient 0004 (Acute stroke) secured a bed. Wait time: 0.00 days.
Day 006.05 | Patient 0002 (Acute stroke) discharged to Other. LOS: 3.16 days.
Day 007.22 | Patient 0005 (Complex neurological) arrived.
Day 007.22 | Patient 0005 (Complex neurological) secured a bed. Wait time: 0.00 days.
Day 007.48 | Patient 0006 (Other) arrived.
Day 007.48 | Patient 0006 (Other) secured a bed. Wait time: 0.00 days.
Day 008.13 | Patient 0003 (Comple